# Atividade — Análise do IDHM por Unidade da Federação

Nesta atividade, o arquivo `Tabela4.csv` é analisado com **pandas** e os gráficos são feitos com **matplotlib**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Tabela4.csv", sep=";", decimal=",", encoding="latin1", skiprows=1)

# Remove colunas totalmente vazias
df = df.dropna(axis=1, how="all")

df.head()

## 1. Estados ordenados pelo maior IDH em 2024

In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
df[anos] = df[anos].apply(pd.to_numeric, errors="coerce")

df_ordenado = df.sort_values("2024", ascending=False)
df_ordenado[["Sigla", "Estado", "2024"]]

## 2. Qual estado teve a maior melhora entre 1991 e 2024?

In [ ]:
df["Melhora_1991_2024"] = df["2024"] - df["1991"]

maior_melhora = df.loc[df["Melhora_1991_2024"].idxmax()]
print(
    f'O estado com maior melhora foi {maior_melhora["Estado"]} '
    f'({maior_melhora["Sigla"]}), com aumento de '
    f'{maior_melhora["Melhora_1991_2024"]:.3f} no IDH.'
)

## 3. Existe algum estado em que o IDH piorou?

In [ ]:
piorou = df[df["2024"] < df["1991"]][["Sigla", "Estado", "1991", "2024", "Melhora_1991_2024"]]

if piorou.empty:
    print("Não existe estado em que o IDH de 2024 seja menor que o de 1991.")
else:
    print("Estados em que o IDH piorou:")
    display(piorou)

## 4. Transformando a tabela para o formato longo com `melt`

In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
id_vars = [c for c in df.columns if c not in anos]

df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)

df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")

df_longo.head()

## 5. Gráfico somente de Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(mg["Ano"], mg["IDH"], marker="o", markersize=3, linewidth=1.5)

ax.set_title("Evolução do IDH de Minas Gerais (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Evolução do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(
        grupo["Ano"],
        grupo["IDH"],
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=sigla
    )

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")

fig.tight_layout()
plt.show()

## Respostas

- **Maior IDH em 2024:** os estados estão ordenados na tabela acima, do maior para o menor.
- **Maior melhora entre 1991 e 2024:** **Tocantins (TO)**, que passou de **0,369** para **0,797**, uma melhora de **0,428**.
- **Algum estado piorou?** **Não.** Todos os estados apresentaram IDH de 2024 maior que o de 1991.
